In [1]:
class QuadraticFp:
    """
    Represents elements of the form:
        a + b*i
    where i^2 = 4*i + 99; means i^2 + 97*i + 2=0
    """

    def __init__(self, a, b, p):
        self.p = p
        self.a = a %p  # real part
        self.b = b %p  # coefficient of i

    def __repr__(self):
        return f"{self.a} + {self.b}i"

    def __add__(self, other):
        return QuadraticFp(self.a + other.a,
                          self.b + other.b, 
                          self.p)
    def __eq__(self, other):
        if isinstance(other, QuadraticFp):
            return (
                self.p == other.p and
                self.a == other.a and
                self.b == other.b
            )
    
        if isinstance(other, int):
            return self.a == other % self.p and self.b == 0
    
        return False
    def __neg__(self):
        return QuadraticFp(
            (-self.a) % self.p,
            (-self.b) % self.p,
            self.p
        )

    @classmethod
    def zero(cls, p):
        return cls(0, 0, p)

    @classmethod
    def one(cls, p):
        return cls(1,0,p)
    
    def __sub__(self, other):
        return QuadraticFp(self.a - other.a,
                          self.b - other.b, 
                         self.p)
    def __neg__(self):
        return QuadraticFp(-self.a, -self.b, self.p)

    def __mul__(self, other):
        """
        (a + bi)(c + di)
        = ac + adi + bci + bd i^2

        Replace i^2 = 4i + 99
        """
        a, b = self.a, self.b
        c, d = other.a, other.b
        p = self.p

        real_part = a*c + b*d*99 %p
        i_part = a*d + b*c + 4*b*d %p

        return QuadraticFp(real_part, i_part, p)
    
    def __pow__(self, n):
        result = QuadraticFp(1, 0, self.p)
        base = self
    
        while n:
            if n & 1:
                result = result * base
            base = base * base
            n >>= 1
    
        return result
    def inverse(self):
            """
            Compute multiplicative inverse.
    
            If:
                x = a + b i
    
            We find inverse using the norm:
    
                N(x) = x * conjugate(x)
    
            For polynomial:
                i^2 - 4i - 99 = 0
    
            Conjugate root satisfies:
                i' = 4 - i
    
            So conjugate of (a + b i) is:
                a + b(4 - i)
                = (a + 4b) - b i
            """
    
            p = self.p
            a, b = self.a, self.b
    
            # conjugate
            conj = QuadraticFp(a + 4*b, -b, p)
    
            # norm = x * conjugate (must lie in F_p)
            norm = (self * conj).a  # imaginary part cancels
    
            if norm == 0:
                raise ZeroDivisionError("Element not invertible")
    
            norm_inv = pow(norm, p-2, p)
    
            return QuadraticFp(
                conj.a * norm_inv,
                conj.b * norm_inv,
                p
            )
p=101;
ZERO = QuadraticFp.zero(p)
ONE  = QuadraticFp.one(p)
TWO = QuadraticFp(2,0,p)
THREE = QuadraticFp(3,0,p)
a=ONE
b=QuadraticFp(9,0,p)
E=[a,b]

In [2]:
def mod_inv(n, p):
    return pow(n, p - 2, p)


def is_on_curve(P, a, b):
    """Check whether a point lies on the curve."""
    if P is None:
        return True
    
    x, y = P
    return (y * y - (x * x * x + a * x + b)) == ZERO
def point_neg(P):
    """
    Compute -P on elliptic curve over F_p.
    """
    if P is None:
        return None

    x, y = P
    return (x, (-y))

def scalar_mul(k, P, a):
    """
    Compute k * P using double-and-add.
    Works for positive and negative scalars.
    """

    if k == 0 or P is None:
        return None

    # Handle negative scalar
    if k < 0:
        return scalar_mul(-k, point_neg(P), a)

    result = None  # point at infinity
    addend = P

    while k:
        if k & 1:
            result = point_add(result, addend, a)

        addend = point_double(addend, a)
        k >>= 1

    return result
    

def point_add(P, Q, a):
    """
    Add two points P and Q on elliptic curve over F_p.
    Curve: y^2 = x^3 + ax + b
    """
    # Handle point at infinity
    if P is None:
        return Q
    if Q is None:
        return P

    x1, y1 = P
    x2, y2 = Q

    # P + (-P) = O
    if x1 == x2 and (y1 != y2):
        return None

    # If points are equal -> doubling
    if P == Q:
        if y1==0:
            return None;
        else:
            return point_double(P, a, p)

    # Regular addition
    slope = (y2 - y1) * (x2-x1).inverse() #mod_inv(x2 - x1, p)) % p

    x3 = (slope * slope - x1 - x2)
    y3 = (slope * (x1 - x3) - y1)

    return (x3, y3)


def point_double(P, a):
    """
    Double a point P on elliptic curve over F_p.
    """
    if P is None:
        return None

    x, y = P

    # Tangent is vertical
    if y == 0:
        return None

    slope = (THREE * x * x + a) * (TWO * y).inverse()

    x3 = (slope * slope - TWO * x)
    y3 = (slope * (x - x3) - y)

    return (x3, y3)

In [3]:
def line(self, R, Q , a):
    # Lets the infinity point of the elliptic curve is INF
        if Q== None:
            raise ValueError("Q must be not infinity of EC.")

        if self == None or R == None:
            if self == R:
                #print("The case of P=R=O", ONE)
                return ONE
            if self == None:
                #print("The case of P=O: Q[0] - R[0]", Q[0] - R[0])
                return Q[0] - R[0]
            if R == None:
                #print("The case of R=O: Q[0] - self[0]", Q[0] - self[0])
                return Q[0] - self[0]
        elif self != R:
            if self[0] == R[0]:
                #print("The case of P<>R, xP=xR: xQ = ", Q[0], " xS = ", self[0], "Result = ", Q[0] - self[0])
                return Q[0] - self[0]
            else:
                l = (R[1] - self[1])*(R[0] - self[0]).inverse()
                #print("The case of P<>R: slopeT-P, yT= ", R[1], "yP= ", self[1], "xT = ", R[0], "xP = ", self[0],  "Q[1] = " ,Q[1], "self[1]= ", self[1], "slope = ", l, "xQ= ", Q[0], "xP= ", self[0], "   Result = ", Q[1] - self[1] - l * (Q[0] - self[0]))
                return Q[1] - self[1] - l * (Q[0] - self[0])
        else:
            numerator = (THREE*self[0]**2 + a)
            denominator = (TWO*self[1])
            if denominator == 0:
                #print("The case of yP=0: Q[0] - self[0]", Q[0] - self[0])
                return Q[0] - self[0]
            else:
                l = numerator*denominator.inverse()
                #print("The case of P=R: yQ = " ,Q[1], "yP= ", self[1], "slopeT-T = " , l, "xQ= ", Q[0], "x=", self[0], "   Result = ", Q[1] - self[1] - l * (Q[0] - self[0]))
                return Q[1] - self[1] - l * (Q[0] - self[0])
def miller(self, Q, n, a):
        if Q == None:
            raise ValueError("Q must be not infinity of EC.")
        if n == 0:
            raise ValueError("n must be nonzero.")
        n_is_negative = False
        if n < 0:
            n = -1 *n;
            n_is_negative = True

        one = 1;
        t = ONE
        V = self
        bitt_n =bin(n)[2:]
        nbin = [int(bitt_n[-i]) for i in range(1, n.bit_length()+1)];
        #print("Bits of n =", nbin)
        i = len(nbin) - 2 
        while i > -1:
            #print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
            #print(" i = ", i)
            S = point_double(V, a) #S = 2*V
            #print("S = ", S)
            ell = line(V, V, Q , a) #ell = V._line_(V, Q)
            #print("The numerator of l = ", ell)
            vee = line(S, point_neg(S), Q , a) #vee = S._line_(-S, Q)
            #print("the denominator of l = ", vee)
            t = (t**2)*(ell*vee.inverse())
            #print("f = ", t)
            V = S
            #print("T = ", V)
            if nbin[i] == 1:
                #print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
                #print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
                #print("The bit is 1 (inside if condition)")
                S = point_add(V, self, a) # S = V+self
                #print("S = (inside if condition)", S)
                ell = line(V,self, Q, a) #ell = V._line_(self, Q)
                #print("The numerator of l (inside if condition)= ", ell)
                vee = line(S, point_neg(S), Q, a) #vee = S._line_(-S, Q)
                #print("the denominator of l (inside if condition)= ", vee)
                t = t*ell*vee.inverse()
                #print("f (inside if condition)= ", t)
                V = S
                #print("T (inside if condition)= ", V)
            i = i-1
        if n_is_negative:
            vee=line(V, point_neg(V), Q)#vee = V._line_(-V, Q)
            t = (t*vee).inverse()
        return t

In [4]:
#line(self, R, Q , a)
V=[QuadraticFp(4,0,p), QuadraticFp(28,0,p)]
self = [QuadraticFp(4,0,p), QuadraticFp(73,0,p)]
Q = [QuadraticFp(4,0,p), QuadraticFp(73,0,p)]
line(V,self, Q, a)

0 + 0i

In [5]:
def weil_pairing(P, Q, n, E):
        a,b = E[0], E[1];
        if is_on_curve(Q, a, b) == False:
            raise ValueError("points must both be on the same curve")
        # Test if P, Q are both in E[n]
        if scalar_mul(n, P, a) !=None or scalar_mul(n, Q, a)!=None:
            raise ValueError("points must both be n-torsion")

        one = 1;

        # Case where P = Q
        if P == Q:
            return one

        # Case where P = O or Q = O
        if P==None or Q == None:
            return one

        try:
            last_bit= n % 2;
            deno = miller(Q, P, n, a)
            res = QuadraticFp(-1,0,p)**last_bit*(miller(P, Q, n, a)*deno.inverse())
            return res
        except ZeroDivisionError:
            return one

def tate_pairing(P, Q, n, k, E):
    a,b = E[0], E[1]
    if is_on_curve(Q, a, b)== False:
        raise ValueError("Points must both be on the same curve")
    #if pow(q,k,n) != 1:
    #    raise ValueError("n does not divide (q^k - 1) for the supplied value of q")
    if scalar_mul(n, P, a)==None:
        raise ValueError("The point P must be n-torsion")
    ePQ = miller(P, Q, n, a)
    exp = int((q**k - 1)/n)
    return ePQ**exp

In [7]:
#comm_L_alpha_g1 =  (65 : 22 : 1)
#comm_R_beta_g2 =  (71*z2 + 30 : 18*z2 + 68 : 1)
comm_L_alpha_g1 = [QuadraticFp(4,0,p), QuadraticFp(28,0,p)]
comm_R_beta_g2 = [QuadraticFp(2, 49,p), QuadraticFp(53, 48,p)]
weil_pairing(comm_L_alpha_g1, comm_R_beta_g2, 119,E)

5 + 61i

In [9]:
yP = QuadraticFp(85,0,p)
xP = QuadraticFp(48,0,p)
xQ=QuadraticFp(92,53,p)
yQ = QuadraticFp(6,7,p)
P=[xP,yP]
Q=[xQ,yQ]
weil_pairing(P, Q, 119,E)

5 + 61i

In [10]:
xP = QuadraticFp(4,0,p)
yP = QuadraticFp(28,0,p)
xQ=QuadraticFp(2,49,p)
yQ = QuadraticFp(53,48,p)
P=[xP,yP]
Q=[xQ,yQ]
weil_pairing(P, Q, 119,E)

5 + 61i

In [16]:
xP = QuadraticFp(48,0,p)
yP = QuadraticFp(85,0,p)
xQ=QuadraticFp(92,53,p)
yQ = QuadraticFp(6,7,p)
P=[xP,yP]
Q=[xQ,yQ]
p1 = weil_pairing(P, Q, 119,E)
p1

5 + 61i

In [17]:
xP = QuadraticFp(53,0,p)
yP = QuadraticFp(60,0,p)
xQ=QuadraticFp(67, 49,p)
yQ = QuadraticFp(32,84,p)
P=[xP,yP]
Q=[xQ,yQ]
p2 = weil_pairing(P, Q, 119,E)
p2

29 + 15i

In [18]:
xP = QuadraticFp(83,0,p)
yP = QuadraticFp(44,0,p)
xQ=QuadraticFp(10,90,p)
yQ = QuadraticFp(96,58,p)
P=[xP,yP]
Q=[xQ,yQ]
p3 = weil_pairing(P, Q, 119,E)
p3

89 + 86i

In [19]:
p1*p2*p3

5 + 61i

In [21]:
#Test 4
# 98*z2 + 13 : 99*z2 + 78
xP = QuadraticFp(72,0,p)
yP = QuadraticFp(72,0,p)
xQ=QuadraticFp(13,98,p)
yQ = QuadraticFp(78,99,p)
P=[xP,yP]
Q=[xQ,yQ]
p4_11 = weil_pairing(P, Q, 119,E)
p4_11

30 + 51i

In [22]:
#Test 4
# 53*z2 + 92 : 7*z2 + 6
xP = QuadraticFp(48,0,p)
yP = QuadraticFp(85,0,p)
xQ=QuadraticFp(92,53,p)
yQ = QuadraticFp(6,7,p)
P=[xP,yP]
Q=[xQ,yQ]
p4_21 = weil_pairing(P, Q, 119,E)
p4_21

5 + 61i

In [23]:
#Test 4
# 49*z2 + 67 : 84*z2 + 32
xP = QuadraticFp(53,0,p)
yP = QuadraticFp(60,0,p)
xQ=QuadraticFp(67,49,p)
yQ = QuadraticFp(32,84,p)
P=[xP,yP]
Q=[xQ,yQ]
p4_22 = weil_pairing(P, Q, 119,E)
p4_22

29 + 15i

In [24]:
#Test 4
# 90*z2 + 10 : 58*z2 + 96
xP = QuadraticFp(24,0,p)
yP = QuadraticFp(11,0,p)
xQ=QuadraticFp(10,90,p)
yQ = QuadraticFp(96,58,p)
P=[xP,yP]
Q=[xQ,yQ]
p4_23 = weil_pairing(P, Q, 119,E)
p4_23

41 + 31i

In [25]:
p4_21*p4_22*p4_23

30 + 51i

In [6]:
# (68*z2 + 74 : 52*z2 + 75) # 63*z2 + 68 : 67*z2 + 46 
yP = QuadraticFp(73,0,p)
xP = QuadraticFp(4,0,p)
xQ=QuadraticFp(92,53,p)
yQ = QuadraticFp(6,7,p)
P=[xP,yP]
Q=[xQ,yQ]
point_add(P,Q,a)
#(99*z2 + 63 : 12*z2 + 14 : 1)

(11 + 76i, 5 + 36i)

In [7]:
point_double(P,a)
#(63*z2 + 19 : 82*z2 + 49 : 1)
# 46*z2 + 6 : 23*z2 + 73

(48 + 0i, 16 + 0i)

In [8]:
n= 119;
print('P = ', P)
print('Q = ', Q)
miller(P, Q, n, a)

P =  [4 + 0i, 73 + 0i]
Q =  [92 + 53i, 6 + 7i]


31 + 81i

In [9]:
miller(Q, P, n, a)

0 + 14i

In [10]:
miller(Q,P, n, a)

0 + 14i

In [11]:
a

1 + 0i

In [12]:
comm_L_alpha_g1 = [QuadraticFp(65,0,p), QuadraticFp(22,0,p)]
comm_R_beta_g2 = [QuadraticFp(30, 71,p), QuadraticFp(68, 18,p)]
weil_pairing(comm_L_alpha_g1, comm_R_beta_g2, 119,E)

41 + 31i

In [25]:
P_first_of_second_pairing = [QuadraticFp(27, 0, p), QuadraticFp(23, 0, p)]
G2 = [QuadraticFp(89,51,p), QuadraticFp(63,93,p)]
alpha_g1 = [QuadraticFp(48,0,p), QuadraticFp(85,0,p)]
beta_g2 = [QuadraticFp(92,53,p), QuadraticFp(6,7,p)]
part1 = weil_pairing(P_first_of_second_pairing, G2, 119,E)
part2 = weil_pairing(alpha_g1, beta_g2, 119,E)
print(part1,part2 ,part1*part2) 

53 + 78i 5 + 61i 41 + 31i


In [26]:
weil_pairing(P, Q, n, E)

98 + 48i

In [28]:
comm_L_alpha_r_delta_g1 = [QuadraticFp(4,0,p), QuadraticFp(28,0,p)]
comm_R_beta_r_delta_g2 = [QuadraticFp(20,14,p), QuadraticFp(84,79,p)]
weil_pairing(comm_L_alpha_r_delta_g1, comm_R_beta_r_delta_g2, 119, E)

88 + 53i

In [30]:
alpha_g1 = [QuadraticFp(48,0,p), QuadraticFp(85, 0, p)]
beta_g2 = [QuadraticFp(92, 53, p), QuadraticFp(6, 7, p)]
P_pub_of_second_pairing = [QuadraticFp(53, 0, p), QuadraticFp(60, 0, p)]
gammaG2 = [QuadraticFp(67,49, p), QuadraticFp(32, 84, p)]
P_priv_of_second_pairing =[QuadraticFp(42,0, p), QuadraticFp(45,0, p)]
deltaG2 = [QuadraticFp(10,90, p), QuadraticFp(96,58, p)]
part1 = weil_pairing(alpha_g1, beta_g2,119, E)
part2 = weil_pairing(P_pub_of_second_pairing, gammaG2, 119, E)
part3 = weil_pairing(P_priv_of_second_pairing, deltaG2, 119, E)
print(part1, part2, part3, part1*part2*part3)

5 + 61i 29 + 15i 62 + 23i 88 + 53i
